# 안쪽 원판을 어둡게 하면 바깥 테두리 검출이 회복될까?

b02 32~35번은 800 입력에서 선택 박스가 밝은 면 쪽으로 작아졌던 연속 장면입니다. 같은 best.pt와 800 입력으로 **원본 / 안쪽 밝은 면 감산 / 천장 감산**을 비교합니다. 목표는 기존 바깥 박스와의 IoU 0.50 회복입니다. 640 비교는 33번 노트북에서 확인할 수 있습니다.

In [ ]:
from datetime import datetime
from pathlib import Path
import hashlib
import json
import matplotlib.pyplot as plt
import pandas as pd
import torch
import ultralytics
from IPython.display import display
from wlcv.config import ROOT
from wlcv.evaluation.plate_resolution import WEIGHTS
from wlcv.evaluation.face_brightness import darkened_frames, run_face_brightness, show_frame
from wlcv.config import VIDEOS
from wlcv.data.frames import read_rgb

pairs = pd.read_csv(ROOT / "outputs/plate_resolution/2026-11-24/paired_frames.csv")
amount = 40  # 8비트 각 RGB 채널에서 최대 40단계 감산

## 먼저 변경 영역 보기
원판의 수동 전체 박스 안에서 회색조 밝기 130을 넘는 밝은 면을 찾습니다. 그 안의 픽셀을 어둡게 하고, 경계 안쪽 5 px에서는 변화량을 서서히 줄입니다. 대조 조건에서는 같은 감산 지도를 오른쪽 천장으로 평행 이동합니다. 두 위치에서 실제 변화량이 일치하도록 픽셀별 감산량을 제한합니다. 합성 영상의 국소 대비와 질감도 함께 달라질 수 있습니다.

In [ ]:
video = "b02_sn_h_02"
index = 33
ref = pairs.query("video_id == @video and frame_index == @index and method == 'Past link'").iloc[0]
outer = ref[["gt_x1", "gt_y1", "gt_x2", "gt_y2"]].to_numpy(float)
images, _, _ = darkened_frames(read_rgb(VIDEOS / f"{video}.mp4", index), outer, amount)
fig, axes = plt.subplots(1, 3, figsize=(8, 4.6), layout="constrained")
for ax, (name, image) in zip(axes, images.items()):
    ax.imshow(image)
    ax.set_title(name)
    ax.axis("off")
display(fig)
plt.close(fig)

## 같은 이력에서 추론
영상 0번부터 현재 프레임 직전까지는 항상 원본입니다. 현재 한 장만 교체하므로, 앞선 합성 프레임에서 생긴 오차가 다음 프레임으로 넘어가지 않습니다. 신뢰도 하한 0.001, 직접 검출 0.10, 시간 연결 반경 0.5배를 그대로 적용합니다. 기준선 재현 표는 33번 원본 결과와 이번 원본 결과를 대조합니다.

In [ ]:
result = run_face_brightness(pairs, amount)
output = ROOT / "outputs/plate_face_brightness" / datetime.now().strftime("%Y%m%d-%H%M%S-%f")
output.mkdir(parents=True)
result.to_csv(output / "comparison.csv", index=False)
settings = dict(weights_sha256=hashlib.sha256(WEIGHTS.read_bytes()).hexdigest(), imgsz=800,
                device="cpu", torch_version=torch.__version__, ultralytics=ultralytics.__version__,
                cutoff=130, max_decrement=amount, ceiling_center=[555, 170], feather_px=5)
(output / "settings.json").write_text(json.dumps(settings, ensure_ascii=False, indent=2))
base = pairs.query("video_id == @video and method == 'Past link'")[["frame_index", "iou_800", "source_800"]]
check = result.query("condition == 'original'").merge(base, on="frame_index", validate="one_to_one")
display(check[["frame_index", "iou_800", "iou_outer", "source_800", "source"]].round(4))

## 선택된 박스와 전체 후보 비교
`iou_outer`는 선택한 박스와 바깥 원판 수동 박스의 겹침, `iou_inner`는 원본에서 근사한 안쪽 밝은 면 박스와의 겹침입니다. `max_candidate_outer_iou`는 정답을 사후 대조해 구한 모든 후보 중 최고 IoU입니다. 정답 박스는 후보 선택에 사용하지 않습니다.

In [ ]:
display(result[["frame_index", "condition", "confidence", "source", "iou_outer", "iou_inner",
                "width_px", "max_candidate_outer_iou", "changed_pixels", "mean_decrement"]].round(4))
for index in [33, 35]:
    fig = show_frame(pairs, result, index)
    fig.savefig(output / f"frame{index}.png", dpi=140)
    display(fig)
    plt.close(fig)
print(output)